<div style="
/* background:linear-gradient(135deg,#061E29 0%,#071F2B 30%,#0A4050 68%,#0D6875 100%); */
/* border:2px solid #79E7DD; */
border-radius:14px;
padding:22px 27px;
margin:20px 8px 14px;
box-shadow:
    /* 0 0 8px rgba(121,231,221,.65), */
    /* 0 0 20px rgba(66,191,183,.25), */
    /* inset 0 0 15px rgba(121,231,221,.05); */
">

<div style="
text-align:center;
color:#79E7DD;
font-size:9px;
font-weight:800;
letter-spacing:2px;
text-transform:uppercase;
">
TASK 3 · Serving predictions on a web interface
</div>

<div style="
text-align:center;
/* color:#A8F2EC; */
font-family:Georgia,'Times New Roman',serif;
font-size:28px;
font-weight:750;
line-height:1.25;
margin-top:6px;
/* text-shadow:0 0 9px rgba(121,231,221,.32); */
">
Web Interface Insights
The dashboard provides two major predictions:
</div>

<div style="
width:110px;
height:3px;
margin:10px auto 9px;
border-radius:20px;
background:#79E7DD;
box-shadow:
    /* 0 0 11px rgba(121,231,221,.82), */
    /* 0 0 20px rgba(66,191,183,.30); */
"></div>

<div style="
text-align:center;
color:#C7F5F1;
font-size:10.4px;
font-weight:500;
line-height:1.6;
max-width:1150px;
margin:auto;
">
Predicted Sales
Helps estimate expected revenue for a selected store and date.
</div>

</div>

<div style="
margin:-3px 12px 12px;
background:#E7F8F6;
/* border:1px solid #79E7DD; */
/* border-left:4px solid #0D6875; */
border-radius:9px;
padding:11px 15px;
">

<div style="
color:#0D6875;
font-size:8.6px;
font-weight:800;
letter-spacing:1.25px;
text-transform:uppercase;
margin-bottom:5px;
">
Predicted Customers
</div>

<div style="
color:#163F48;
font-size:10.8px;
font-weight:500;
line-height:1.65;
">


Helps estimate expected customer traffic.
These two predictions complement each other.

</div>
</div>

<h2><span style="color:lightblue"><b>Train and save both models</b></span></h2>

Your existing Random Forest pipeline can be extended to predict both targets.

In [11]:
import joblib
import pandas as pd
from datetime import datetime
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
# Load the training data if train_df has not already been created
if "train_df" not in globals():
    search_locations = [
        Path.cwd(),
        Path.home() / "Desktop" / "Project 6",
        Path.home() / "OneDrive" / "Desktop" / "Project 6",
        Path.home(),
    ]

    train_path = None

    for location in search_locations:
        if location.exists():
            matches = list(location.rglob("train.csv"))
            if matches:
                train_path = matches[0]
                break

    if train_path is None:
        raise FileNotFoundError(
            "Could not find train.csv. Place it in the notebook folder "
            "or in your Desktop\\Project 6 folder."
        )

    train_df = pd.read_csv(train_path)

# Ensure the feature lists are defined in this notebook
numeric_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
]

categorical_features = [
    "StateHoliday",
]

# Define training features and targets before fitting the models.
feature_columns = list(dict.fromkeys(numeric_features + categorical_features))
missing_columns = [column for column in feature_columns if column not in train_df.columns]

if missing_columns:
    raise KeyError(f"Feature columns not found in training data: {missing_columns}")

if "Sales" not in train_df.columns or "Customers" not in train_df.columns:
    raise KeyError("The training data must contain 'Sales' and 'Customers' columns.")

X_train = train_df[feature_columns].copy()

# Ensure categorical columns contain one consistent data type.
for column in categorical_features:
    X_train[column] = X_train[column].astype(str)

y_train = train_df["Sales"]

# The preprocessor must be created before any model uses it.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# -----------------------------------------
# Sales model
# -----------------------------------------
sales_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=20,
                min_samples_split=5,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

sales_model.fit(
    X_train,
    y_train,
)

# -----------------------------------------
# Customers model
# -----------------------------------------
customers_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=20,
                min_samples_split=5,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

customers_model.fit(
    X_train,
    train_df["Customers"],
)

# -----------------------------------------
# Save models
# -----------------------------------------
timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M-%S-%f")[:-3]

joblib.dump(
    sales_model,
    f"sales_model_{timestamp}.pkl",
)

joblib.dump(
    customers_model,
    f"customers_model_{timestamp}.pkl",
)

print("Both models saved successfully.")

Both models saved successfully.


<h2><span style="color:lightblue"><b>ROSSMANN STORE SALES + CUSTOMER PREDICTION DASHBOARD</b></span></h2>

Combined Sales & Customer Trend

Combining both metrics helps managers understand whether changes in revenue are driven by customer volume or other factors.

In [17]:


import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from datetime import datetime


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Rossmann Prediction Dashboard",
    page_icon="🏪",
    layout="wide"
)


# ============================================================
# LOAD MODELS
# ============================================================

@st.cache_resource
def load_models():

    sales_model = joblib.load(
        "sales_model_15-09-2026-03-38-47-642.pkl"
    )

    customers_model = joblib.load(
        "customers_model_15-09-2026-03-38-47-642.pkl"
    )

    return sales_model, customers_model


sales_model, customers_model = load_models()


# ============================================================
# HEADER
# ============================================================

st.title(
    "🏪 Rossmann Store Prediction Dashboard"
)

st.markdown(
    """
    ### Sales & Customer Forecasting

    This dashboard allows store managers to enter store and
    date-related information and obtain predicted:

    - 💰 Daily Sales
    - 👥 Customer Numbers
    """
)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header(
    "Store Information"
)

store_id = st.sidebar.number_input(
    "Store ID",
    min_value=1,
    max_value=2000,
    value=1,
    step=1
)

store_type = st.sidebar.selectbox(
    "Store Type",
    ["a", "b", "c", "d"]
)

assortment = st.sidebar.selectbox(
    "Assortment",
    ["a", "b", "c"]
)

competition_distance = st.sidebar.number_input(
    "Competition Distance",
    min_value=0.0,
    value=1000.0
)

competition_month = st.sidebar.number_input(
    "Competition Open Month",
    min_value=0,
    max_value=12,
    value=0
)

competition_year = st.sidebar.number_input(
    "Competition Open Year",
    min_value=0,
    max_value=2026,
    value=0
)

promo2 = st.sidebar.selectbox(
    "Promo2",
    [0, 1]
)

promo2_week = st.sidebar.number_input(
    "Promo2 Since Week",
    min_value=0,
    max_value=52,
    value=0
)

promo2_year = st.sidebar.number_input(
    "Promo2 Since Year",
    min_value=0,
    max_value=2026,
    value=0
)

promo_interval = st.sidebar.selectbox(
    "Promo Interval",
    [
        "None",
        "Jan,Apr,Jul,Oct",
        "Feb,May,Aug,Nov",
        "Mar,Jun,Sept,Dec"
    ]
)


# ============================================================
# INPUT METHOD
# ============================================================

st.sidebar.header(
    "Prediction Input"
)

input_method = st.sidebar.radio(
    "Choose Input Method",
    [
        "Manual Input",
        "Upload CSV"
    ]
)


# ============================================================
# FUNCTION: FEATURE ENGINEERING
# ============================================================

def create_features(
    data,
    store_id,
    store_type,
    assortment,
    competition_distance,
    competition_month,
    competition_year,
    promo2,
    promo2_week,
    promo2_year,
    promo_interval
):

    df = data.copy()

    df["Date"] = pd.to_datetime(
        df["Date"]
    )

    # -----------------------------------------
    # Date features
    # -----------------------------------------

    df["Year"] = df["Date"].dt.year

    df["Month"] = df["Date"].dt.month

    df["Day"] = df["Date"].dt.day

    df["WeekOfYear"] = (
        df["Date"]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    df["Quarter"] = (
        df["Date"]
        .dt.quarter
    )

    df["Weekday"] = (
        df["Date"]
        .dt.weekday
    )

    df["DayOfWeek"] = (
        df["Weekday"] + 1
    )

    # -----------------------------------------
    # Weekend
    # -----------------------------------------

    df["IsWeekend"] = (
        df["Weekday"] >= 5
    ).astype(int)

    # -----------------------------------------
    # Month position
    # -----------------------------------------

    df["MonthPosition"] = np.where(
        df["Day"] <= 10,
        "Beginning",
        np.where(
            df["Day"] <= 20,
            "Middle",
            "End"
        )
    )

    df["IsBeginningOfMonth"] = (
        df["Day"] <= 10
    ).astype(int)

    df["IsMidMonth"] = (
        (df["Day"] > 10) &
        (df["Day"] <= 20)
    ).astype(int)

    df["IsEndOfMonth"] = (
        df["Day"] > 20
    ).astype(int)

    # -----------------------------------------
    # Cyclic features
    # -----------------------------------------

    df["Day_sin"] = np.sin(
        2 * np.pi * df["Day"] / 31
    )

    df["Day_cos"] = np.cos(
        2 * np.pi * df["Day"] / 31
    )

    df["Month_sin"] = np.sin(
        2 * np.pi * df["Month"] / 12
    )

    df["Month_cos"] = np.cos(
        2 * np.pi * df["Month"] / 12
    )

    df["Weekday_sin"] = np.sin(
        2 * np.pi * df["Weekday"] / 7
    )

    df["Weekday_cos"] = np.cos(
        2 * np.pi * df["Weekday"] / 7
    )

    # -----------------------------------------
    # Store information
    # -----------------------------------------

    df["Store"] = store_id

    df["StoreType"] = store_type

    df["Assortment"] = assortment

    df["CompetitionDistance"] = (
        competition_distance
    )

    df["CompetitionOpenSinceMonth"] = (
        competition_month
    )

    df["CompetitionOpenSinceYear"] = (
        competition_year
    )

    df["Promo2"] = promo2

    df["Promo2SinceWeek"] = (
        promo2_week
    )

    df["Promo2SinceYear"] = (
        promo2_year
    )

    df["PromoInterval"] = (
        promo_interval
    )

    # -----------------------------------------
    # Competition age
    # -----------------------------------------

    df["CompetitionAge"] = np.where(
        competition_year > 0,
        df["Year"] - competition_year,
        0
    )

    df["CompetitionAge"] = (
        df["CompetitionAge"]
        .clip(lower=0)
    )

    df["HasCompetition"] = (
        competition_distance > 0
    ).astype(int)

    # -----------------------------------------
    # Promo2 age
    # -----------------------------------------

    df["HasPromo2"] = (
        promo2 == 1
    ).astype(int)

    df["Promo2Age"] = np.where(
        promo2_year > 0,
        df["Year"] - promo2_year,
        0
    )

    df["Promo2Age"] = (
        df["Promo2Age"]
        .clip(lower=0)
    )

    # -----------------------------------------
    # Holiday
    # -----------------------------------------

    if "IsHoliday" in df.columns:

        df["IsStateHoliday"] = (
            df["IsHoliday"]
            .astype(int)
        )

    elif "StateHoliday" in df.columns:

        df["IsStateHoliday"] = (
            df["StateHoliday"]
            .astype(str)
            .isin(["a", "b", "c"])
            .astype(int)
        )

    else:

        df["IsStateHoliday"] = 0

    if "SchoolHoliday" not in df.columns:

        df["SchoolHoliday"] = 0

    df["IsSchoolHoliday"] = (
        df["SchoolHoliday"]
        .astype(int)
    )

    # -----------------------------------------
    # Promo
    # -----------------------------------------

    if "IsPromo" in df.columns:

        df["Promo"] = (
            df["IsPromo"]
            .astype(int)
        )

    elif "Promo" not in df.columns:

        df["Promo"] = 0

    # -----------------------------------------
    # Open
    # -----------------------------------------

    if "Open" not in df.columns:

        df["Open"] = 1

    # -----------------------------------------
    # Holiday distance
    # -----------------------------------------

    df["DaysToNextHoliday"] = 999

    df["DaysAfterHoliday"] = 999

    # -----------------------------------------
    # Historical values
    #
    # These should ideally come from the
    # historical database automatically.
    # -----------------------------------------

    historical_defaults = {

        "Sales_Lag_1": 5000,

        "Sales_Lag_7": 5000,

        "Sales_Lag_14": 5000,

        "Sales_Lag_28": 5000,

        "Sales_Rolling_Mean_7": 5000,

        "Sales_Rolling_Mean_14": 5000,

        "Sales_Rolling_Mean_28": 5000,

        "Sales_Rolling_Std_7": 1000
    }

    for col, value in historical_defaults.items():

        if col not in df.columns:

            df[col] = value

    return df


# ============================================================
# REQUIRED MODEL FEATURES
# ============================================================

model_features = [

    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",

    "StoreType",
    "Assortment",

    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",

    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",

    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "Quarter",
    "Weekday",

    "IsWeekend",

    "MonthPosition",

    "IsBeginningOfMonth",
    "IsMidMonth",
    "IsEndOfMonth",

    "Day_sin",
    "Day_cos",

    "Month_sin",
    "Month_cos",

    "Weekday_sin",
    "Weekday_cos",

    "CompetitionAge",
    "HasCompetition",

    "HasPromo2",
    "Promo2Age",

    "IsStateHoliday",
    "IsSchoolHoliday",

    "DaysToNextHoliday",
    "DaysAfterHoliday",

    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Lag_28",

    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_28",

    "Sales_Rolling_Std_7"
]


# ============================================================
# MANUAL INPUT
# ============================================================

if input_method == "Manual Input":

    st.header(
        "📅 Manual Prediction"
    )

    selected_date = st.date_input(
        "Prediction Date",
        datetime.today()
    )

    col1, col2, col3 = st.columns(3)

    with col1:

        is_promo = st.selectbox(
            "Is Promo?",
            [0, 1]
        )

    with col2:

        is_holiday = st.selectbox(
            "Is Holiday?",
            [0, 1]
        )

    with col3:

        school_holiday = st.selectbox(
            "School Holiday?",
            [0, 1]
        )

    manual_data = pd.DataFrame({
        "Date": [selected_date],
        "IsPromo": [is_promo],
        "IsHoliday": [is_holiday],
        "SchoolHoliday": [school_holiday]
    })

    if st.button(
        "🔮 Predict Sales & Customers"
    ):

        features_df = create_features(
            manual_data,
            store_id,
            store_type,
            assortment,
            competition_distance,
            competition_month,
            competition_year,
            promo2,
            promo2_week,
            promo2_year,
            promo_interval
        )

        X_prediction = features_df[
            model_features
        ]

        predicted_sales = (
            sales_model
            .predict(X_prediction)
        )

        predicted_customers = (
            customers_model
            .predict(X_prediction)
        )

        predicted_sales = np.maximum(
            predicted_sales,
            0
        )

        predicted_customers = np.maximum(
            predicted_customers,
            0
        )

        result = pd.DataFrame({

            "Date": features_df["Date"],

            "Store_ID": [
                store_id
            ],

            "Predicted_Sales":
                predicted_sales,

            "Predicted_Customers":
                predicted_customers
        })

        # -----------------------------------------
        # KPI
        # -----------------------------------------

        st.subheader(
            "Prediction Result"
        )

        c1, c2 = st.columns(2)

        with c1:

            st.metric(
                "💰 Predicted Sales",
                f"₹{predicted_sales[0]:,.0f}"
            )

        with c2:

            st.metric(
                "👥 Predicted Customers",
                f"{predicted_customers[0]:,.0f}"
            )

        st.dataframe(
            result,
            use_container_width=True
        )


# ============================================================
# CSV INPUT
# ============================================================

else:

    st.header(
        "📁 Upload Prediction CSV"
    )

    st.markdown(
        """
        Upload a CSV containing at least:

        `Date`, `IsHoliday`, `IsWeekend`, `IsPromo`

        Additional date-dependent parameters can also be
        included.
        """
    )

    uploaded_file = st.file_uploader(
        "Upload CSV",
        type=["csv"]
    )

    if uploaded_file is not None:

        input_csv = pd.read_csv(
            uploaded_file
        )

        st.subheader(
            "Uploaded Data"
        )

        st.dataframe(
            input_csv.head(20),
            use_container_width=True
        )

        if st.button(
            "🚀 Generate Predictions"
        ):

            if "Date" not in input_csv.columns:

                st.error(
                    "CSV must contain a Date column."
                )

                st.stop()

            features_df = create_features(
                input_csv,
                store_id,
                store_type,
                assortment,
                competition_distance,
                competition_month,
                competition_year,
                promo2,
                promo2_week,
                promo2_year,
                promo_interval
            )

            X_prediction = features_df[
                model_features
            ]

            # -------------------------------------
            # Sales
            # -------------------------------------

            predicted_sales = (
                sales_model
                .predict(X_prediction)
            )

            # -------------------------------------
            # Customers
            # -------------------------------------

            predicted_customers = (
                customers_model
                .predict(X_prediction)
            )

            predicted_sales = np.maximum(
                predicted_sales,
                0
            )

            predicted_customers = np.maximum(
                predicted_customers,
                0
            )

            # -------------------------------------
            # Results
            # -------------------------------------

            results = pd.DataFrame({

                "Date":
                    features_df["Date"],

                "Store_ID":
                    store_id,

                "Predicted_Sales":
                    predicted_sales,

                "Predicted_Customers":
                    predicted_customers
            })

            # -------------------------------------
            # Display
            # -------------------------------------

            st.subheader(
                "📊 Prediction Results"
            )

            st.dataframe(
                results,
                use_container_width=True
            )

            # -------------------------------------
            # KPI
            # -------------------------------------

            total_sales = (
                results["Predicted_Sales"]
                .sum()
            )

            total_customers = (
                results["Predicted_Customers"]
                .sum()
            )

            avg_sales = (
                results["Predicted_Sales"]
                .mean()
            )

            avg_customers = (
                results["Predicted_Customers"]
                .mean()
            )

            c1, c2, c3, c4 = st.columns(4)

            with c1:

                st.metric(
                    "Total Sales",
                    f"₹{total_sales:,.0f}"
                )

            with c2:

                st.metric(
                    "Total Customers",
                    f"{total_customers:,.0f}"
                )

            with c3:

                st.metric(
                    "Average Daily Sales",
                    f"₹{avg_sales:,.0f}"
                )

            with c4:

                st.metric(
                    "Average Daily Customers",
                    f"{avg_customers:,.0f}"
                )

            # =================================================
            # SALES CHART
            # =================================================

            st.subheader(
                "💰 Predicted Sales"
            )

            fig1, ax1 = plt.subplots(
                figsize=(12, 5)
            )

            ax1.plot(
                results["Date"],
                results["Predicted_Sales"],
                marker="o"
            )

            ax1.set_xlabel(
                "Date"
            )

            ax1.set_ylabel(
                "Predicted Sales"
            )

            ax1.set_title(
                "Predicted Daily Sales"
            )

            plt.xticks(
                rotation=45
            )

            plt.tight_layout()

            st.pyplot(fig1)

            # =================================================
            # CUSTOMER CHART
            # =================================================

            st.subheader(
                "👥 Predicted Customers"
            )

            fig2, ax2 = plt.subplots(
                figsize=(12, 5)
            )

            ax2.plot(
                results["Date"],
                results["Predicted_Customers"],
                marker="o"
            )

            ax2.set_xlabel(
                "Date"
            )

            ax2.set_ylabel(
                "Predicted Customers"
            )

            ax2.set_title(
                "Predicted Daily Customers"
            )

            plt.xticks(
                rotation=45
            )

            plt.tight_layout()

            st.pyplot(fig2)

            # =================================================
            # COMBINED CHART
            # =================================================

            st.subheader(
                "📈 Sales & Customer Forecast"
            )

            fig3, ax1 = plt.subplots(
                figsize=(13, 6)
            )

            ax1.plot(
                results["Date"],
                results["Predicted_Sales"],
                marker="o",
                label="Sales"
            )

            ax1.set_xlabel(
                "Date"
            )

            ax1.set_ylabel(
                "Predicted Sales"
            )

            ax2 = ax1.twinx()

            ax2.plot(
                results["Date"],
                results["Predicted_Customers"],
                marker="s",
                label="Customers"
            )

            ax2.set_ylabel(
                "Predicted Customers"
            )

            ax1.set_title(
                "Predicted Sales and Customers"
            )

            plt.xticks(
                rotation=45
            )

            plt.tight_layout()

            st.pyplot(fig3)

            # =================================================
            # DOWNLOAD
            # =================================================

            csv_data = results.to_csv(
                index=False
            )

            st.download_button(
                label="⬇️ Download Predictions CSV",

                data=csv_data,

                file_name=(
                    "rossmann_predictions.csv"
                ),

                mime="text/csv"
            )

2026-09-15 03:58:10.120 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 03:58:10.122 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 03:58:11.279 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 03:58:11.482 
  command:

    streamlit run C:\Users\OM SHAH\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-15 03:58:11.484 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 03:58:11.485 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 03:58:11.48

In [23]:
prediction_results = st.session_state.get(
    "prediction_results",
    globals().get("results", globals().get("result"))
)

if prediction_results is None:
    prediction_input = pd.DataFrame({
        "Store": [store_id],
        "DayOfWeek": [
            pd.to_datetime(manual_data["Date"]).dt.dayofweek.iloc[0] + 1
        ],
        "Open": [1],
        "Promo": [is_promo],
        "SchoolHoliday": [school_holiday],
        "StateHoliday": [str(is_holiday)],
    })

    predicted_sales = sales_model.predict(
        prediction_input[feature_columns]
    )

    predicted_customers = customers_model.predict(
        prediction_input[feature_columns]
    )

    prediction_results = pd.DataFrame({
        "Date": manual_data["Date"],
        "Store_ID": [store_id],
        "Predicted_Sales": predicted_sales,
        "Predicted_Customers": predicted_customers,
    })

    st.session_state["prediction_results"] = prediction_results

csv_data = prediction_results.to_csv(index=False)

st.download_button(
    label="⬇️ Download Predictions CSV",
    data=csv_data,
    file_name="rossmann_predictions.csv",
    mime="text/csv",
)


2026-09-15 04:08:50.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.697 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.698 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.707 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.709 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.711 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 04:08:50.713 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

False

<h2><span style="color:lightblue"><b>Hosting & Deployment</b></span></h2>

The application can be hosted using a cloud platform such as Streamlit Community Cloud.
The final application provides a shareable URL that can be submitted as the project deployment link.

<h2><span style="color:lightblue"><b>Overall Project Insights</b></span></h2>

The complete project provides several important business insights.

1 – Historical demand is critical
Previous sales and rolling sales patterns provide strong information about future demand.
Therefore, maintaining accurate historical sales data is essential.

2 – Promotions affect demand
Promotional campaigns should be considered when forecasting sales and customer traffic.
Managers can use the predictions to prepare inventory before promotional periods.

3 – Calendar patterns matter
Weekdays, weekends, months, holidays and seasonal periods influence store performance.
Sales planning should therefore be calendar-aware.

4 – Store characteristics matter
Different stores have different sales potential because of:
Store type, Assortment, Competition ,Location-related characteristics
A single forecasting strategy may not be equally accurate for every store.

5 – Customer prediction adds operational value
Predicting customers in addition to sales enables better:
Workforce planning, Inventory management, Store capacity planning, Promotional planning


6 – Uncertainty should be considered
Managers should not treat every prediction as exact.
Predictions with higher uncertainty should receive additional operational attention.






<h2><span style="color:lightblue"><b>Business Recommendations</b></span></h2>

1 – Inventory Optimization
Use predicted sales to determine expected inventory requirements.
For high predicted-sales days: Increase inventory, Prepare additional stock, Reduce stock-out risk
For low predicted-sales days:Avoid excessive inventory, Reduce potential wastage, Optimize replenishment.


2 – Promotion Planning
Use the model to compare expected demand during promotional and non-promotional periods.
Promotions should be concentrated around periods where they are expected to generate meaningful incremental demand.

3 – Workforce Planning
Use predicted customer numbers to schedule staff.
High predicted customers
→ More staff
Low predicted customers
→ Lean staffing
This can improve customer service while controlling operational costs.


4 – Store-Specific Strategies
Store managers should not use a single strategy for every location.
Stores should be segmented according to:Sales potential, Customer traffic, Competition, Store type
Historical demand,Promotional response
High-performing stores may require aggressive inventory planning, while weaker stores may require targeted promotional strategies.

5 – Monitor Competition
Competition distance and competition-related features should be continuously monitored.
Stores facing strong nearby competition may require:Targeted promotions, Better assortment, Improved customer experience, Competitive pricing


6 – Use Prediction Intervals for Risk Management
When prediction uncertainty is high, managers should maintain additional safety stock or operational capacity.This is more reliable than planning solely around the point prediction.

7 – Retrain the Model Regularly
Customer behavior and market conditions change over time.
The model should therefore be periodically retrained using recent data.


8 – Monitor Model Performance
After deployment, compare:Actual Sales vs. Predicted Sales
Track:MAE, RMSE, MAPE, R², Prediction bias
This helps detect model degradation.

<h2><span style="color:lightblue"><b>Future Enhancements</b></span></h2>

The current project can be extended further.

1. Automated lag calculation
Instead of fallback values, automatically calculate:
Lag 1
Lag 7
Lag 14
Lag 28
Rolling 7
Rolling 14
Rolling 28
from historical store-level sales.

2. Advanced models
Compare:
XGBoost
LightGBM
CatBoost
Gradient Boosting

3. Store-level forecasting
Build separate or hierarchical forecasting strategies for individual stores.

4. Automated retraining
Create a scheduled pipeline that retrains the model as new sales data becomes available.

5. Model monitoring
Add a production monitoring system for:
Prediction accuracy
Data drift
Feature drift
Model drift


6. Explainable AI
Add SHAP explanations to show why a particular prediction is high or low.

<h2><span style="color:lightblue"><b>Final Conclusion</b></span></h2>


The Rossmann Store Sales Prediction project demonstrates an end-to-end data science and machine learning solution for retail demand forecasting.

The analysis shows that retail sales are influenced by a combination of historical demand, promotions, calendar patterns, holidays, store characteristics and competitive conditions.

Traditional tree-based regression models provide a strong solution for structured Rossmann data, while LSTM provides an alternative approach for learning sequential time-series patterns.

The Streamlit dashboard converts the analytical model into a practical business application. Store managers can enter store and date information, upload prediction data, obtain expected sales and customer numbers, visualize demand trends and download the results.

<h2><span style="color:lightblue"><b>Final Business Value</b></span></h2>

The solution can support Rossmann-style retail operations by helping managers:

Forecast sales,
Estimate customer traffic,
Optimize inventory,
Plan staffing,
Improve promotional planning,
Reduce stock-out risk,
Reduce overstocking,
Identify demand patterns,
Make data-driven store decisions.

<h2><span style="color:lightblue"><b>Final Statement</b></span></h2>

Overall, the project demonstrates how machine learning and time-series forecasting can transform historical retail data into actionable business intelligence. The deployed prediction dashboard bridges the gap between data science and real-world decision-making by providing store managers with accessible, data-driven sales and customer forecasts.